# Анализ данных, обучение моделей и метрики

Ноутбук дает рабочий цикл эксперимента: первичный EDA, подготовка train/validation, обучение baseline-моделей, сравнение метрик и разбор ошибок. Обучение выполняется только на train, выбор модели — только на validation. Official test читается только при аудите пересечений и не участвует в обучении или подборе.

Меняйте параметры в ячейке конфигурации и запускайте ноутбук сверху вниз. Каждый запуск сохраняет сводные результаты, отчеты по классам, матрицы ошибок, сериализованные модели и записи MLflow в проекте.

## 1. Подключение проекта и библиотек

In [ ]:
from pathlib import Path
import json
import sys
import time
import joblib
import mlflow
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, precision_recall_fscore_support
import sklearn

def find_project_root(start=Path.cwd()):
    for candidate in (start, *start.parents):
        if (candidate / 'configs' / 'baseline.json').exists() and (candidate / 'src').is_dir():
            return candidate
    raise FileNotFoundError('Не найдена папка проекта с configs/baseline.json')

ROOT = find_project_root()
sys.path.insert(0, str(ROOT / 'src'))
from models.baseline import build_models
CONFIG_PATH = ROOT / 'configs' / 'baseline.json'
DATA_DIR = ROOT / 'data' / 'processed'
OUT_DIR = ROOT / 'experiments' / 'baseline'
MODEL_DIR = ROOT / 'work' / 'notebook_models'
OUT_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)
print(f'Проект: {ROOT}')
print(f'Python: {sys.version.split()[0]} | scikit-learn: {sklearn.__version__}')

## 2. Первичный анализ датасета (EDA)

Ноутбук проверит исходные файлы по SHA-256, покажет пропуски, баланс классов, длины запросов, повторы и пересечения формулировок. Если исходных файлов нет, EDA покажет доступную подготовленную выборку.

In [ ]:
import runpy
raw_train = ROOT / 'data' / 'raw' / 'train.csv'
raw_test = ROOT / 'data' / 'raw' / 'test.csv'
if raw_train.exists() and raw_test.exists():
    eda_module = runpy.run_path(str(ROOT / 'src' / 'data' / 'eda.py'))
    eda_module['main']()
    eda_train_df = pd.read_csv(raw_train)
    eda_summary = json.loads((ROOT / 'reports' / 'eda_summary.json').read_text(encoding='utf-8'))
    display(pd.DataFrame([{'split': name, 'rows': values['rows'], 'classes': values['n_classes'], 'missing': sum(values['missing_by_column'].values()), 'median_chars': values['text_length']['characters']['0.5'], 'normalized_duplicate_rows': values['duplicates']['normalized_duplicate_rows']} for name, values in eda_summary['splits'].items()]))
    display(pd.DataFrame([eda_summary['cross_split']]))
else:
    processed_train_path = ROOT / 'data' / 'processed' / 'train.csv'
    if not processed_train_path.exists():
        raise FileNotFoundError('Для EDA добавьте BANKING77 train.csv и test.csv в data/raw.')
    print('Исходные CSV отсутствуют; строю обзор подготовленной train-выборки.')
    eda_train_df = pd.read_csv(processed_train_path)

class_counts = eda_train_df['category'].value_counts().sort_values()
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
class_counts.plot(kind='barh', ax=axes[0], color='#168A98', title='Число train-примеров по классам')
axes[0].set_xlabel('Примеров')
text_lengths = eda_train_df['text'].astype(str).str.len()
axes[1].hist(text_lengths, bins=30, color='#168A98', edgecolor='white')
axes[1].set_title('Длина запросов в символах')
axes[1].set_xlabel('Символов')
axes[1].set_ylabel('Запросов')
fig.tight_layout()
plt.show()
display(eda_train_df.head())

## 3. Проверка конфигурации и подготовка выборок

Если сырые файлы уже находятся в `data/raw/`, эта ячейка повторно создаст воспроизводимое разбиение. Очистка исключает дубликаты и совпадения с official test по нормализованному тексту; метки test не участвуют в обучении и выборе модели.

In [ ]:
config = json.loads(CONFIG_PATH.read_text(encoding='utf-8-sig'))
raw_train = ROOT / 'data' / 'raw' / 'train.csv'
raw_test = ROOT / 'data' / 'raw' / 'test.csv'
if raw_train.exists() and raw_test.exists():
    import runpy
    split_module = runpy.run_path(str(ROOT / 'src' / 'data' / 'prepare_splits.py'))
    split_module['main']()
else:
    print('Исходные CSV не найдены; использую уже подготовленные data/processed файлы.')

train_path = DATA_DIR / 'train.csv'
validation_path = DATA_DIR / 'validation.csv'
if not train_path.exists() or not validation_path.exists():
    raise FileNotFoundError('Не найдены train.csv/validation.csv. Положите исходные CSV в data/raw и повторите ячейку.')
train_df = pd.read_csv(train_path)
validation_df = pd.read_csv(validation_path)
required_columns = {'text', 'category'}
assert required_columns.issubset(train_df.columns) and required_columns.issubset(validation_df.columns)
assert train_df[list(required_columns)].notna().all().all()
assert validation_df[list(required_columns)].notna().all().all()
assert set(train_df['category']) == set(validation_df['category'])

display(pd.DataFrame({'split': ['train', 'validation'], 'rows': [len(train_df), len(validation_df)], 'classes': [train_df.category.nunique(), validation_df.category.nunique()]}))
display(train_df.head())

## 4. Настройки эксперимента

Параметры ниже можно изменить перед повторным обучением. Модельный pipeline включает TF-IDF внутри fit, чтобы словарь строился только по train.

In [ ]:
experiment_name = config['tracking']['experiment']
tracking_db = ROOT / config['tracking']['database']
mlflow.set_tracking_uri(f"sqlite:///{tracking_db.as_posix()}")
mlflow.set_experiment(experiment_name)

X_train = train_df['text'].astype(str)
y_train = train_df['category'].astype(str)
X_val = validation_df['text'].astype(str)
y_val = validation_df['category'].astype(str)
labels = sorted(y_train.unique())
models = build_models(config)
print(f'Эксперимент MLflow: {experiment_name}')
print(f'Моделей к обучению: {len(models)} | train: {len(X_train):,} | validation: {len(X_val):,}')
display(pd.DataFrame([{'model': name, 'estimator': repr(model)[:180]} for name, model in models.items()]))

## 5. Обучение и оценка

Для каждой модели рассчитываются accuracy, macro/weighted precision, recall и F1. Macro F1 усредняет качество по классам и используется для сортировки. Модели и артефакты сохраняются локально, а параметры и метрики логируются в MLflow.

In [ ]:
results = []
predictions = {}
reports = {}
fitted_models = {}
for name, model in models.items():
    start = time.perf_counter()
    model.fit(X_train, y_train)
    fit_seconds = time.perf_counter() - start
    y_pred = np.asarray(model.predict(X_val))
    predictions[name] = y_pred
    fitted_models[name] = model
    report = classification_report(y_val, y_pred, labels=labels, output_dict=True, zero_division=0)
    reports[name] = pd.DataFrame(report).T
    macro_p, macro_r, macro_f1, _ = precision_recall_fscore_support(y_val, y_pred, labels=labels, average='macro', zero_division=0)
    weighted_p, weighted_r, weighted_f1, _ = precision_recall_fscore_support(y_val, y_pred, labels=labels, average='weighted', zero_division=0)
    metrics = {'accuracy': accuracy_score(y_val, y_pred), 'macro_precision': macro_p, 'macro_recall': macro_r, 'macro_f1': macro_f1, 'weighted_precision': weighted_p, 'weighted_recall': weighted_r, 'weighted_f1': weighted_f1, 'fit_seconds': fit_seconds}
    model_path = MODEL_DIR / f'{name}.joblib'
    joblib.dump(model, model_path)
    reports[name].to_csv(OUT_DIR / f'{name}_classification_report.csv', index_label='label')
    confusion = pd.DataFrame(confusion_matrix(y_val, y_pred, labels=labels), index=labels, columns=labels)
    confusion.to_csv(OUT_DIR / f'{name}_confusion_matrix.csv', index_label='true_label')
    with mlflow.start_run(run_name=name) as run:
        mlflow.log_params({'model': name, 'sklearn_version': sklearn.__version__, 'train_rows': len(X_train), 'validation_rows': len(X_val), 'tfidf_ngram_range': str(config['tfidf']['ngram_range']), 'tfidf_min_df': config['tfidf']['min_df']})
        mlflow.set_tags({'dataset': 'BANKING77', 'evaluation_split': 'validation', 'workflow': 'jupyter_notebook'})
        mlflow.log_metrics({key: float(value) for key, value in metrics.items()})
        mlflow.log_artifact(str(model_path), artifact_path='serialized_model')
        mlflow.log_artifact(str(OUT_DIR / f'{name}_classification_report.csv'), artifact_path='evaluation')
        metrics['mlflow_run_id'] = run.info.run_id
    results.append({'model': name, **{key: float(value) for key, value in metrics.items() if key != 'mlflow_run_id'}, 'mlflow_run_id': metrics['mlflow_run_id']})

results_df = pd.DataFrame(results).sort_values('macro_f1', ascending=False).reset_index(drop=True)
results_df.to_csv(OUT_DIR / 'notebook_results.csv', index=False)
display(results_df.style.format(precision=4).highlight_max(subset=['accuracy', 'macro_f1', 'weighted_f1'], color='#d7f2df'))
print(f'Сводка записана: {OUT_DIR / "notebook_results.csv"}')

## 6. Разбор лучшей модели и ошибок

Выбор лучшей модели выполняется по Macro F1 на validation. После изменения конфигурации или данных снова выполните ячейку обучения перед этим разделом.

In [ ]:
best_name = results_df.iloc[0]['model']
best_model = fitted_models[best_name]
best_pred = predictions[best_name]
print(f'Лучшая модель по validation Macro F1: {best_name} ({results_df.iloc[0]["macro_f1"]:.4f})')
display(reports[best_name].loc[labels].sort_values('f1-score').head(15))

mistakes = validation_df.loc[y_val.to_numpy() != best_pred].copy()
mistakes['predicted_category'] = best_pred[y_val.to_numpy() != best_pred]
mistakes.to_csv(OUT_DIR / 'notebook_best_model_mistakes.csv', index=False)
confusions = mistakes.groupby(['category', 'predicted_category']).size().sort_values(ascending=False).rename('count').reset_index()
display(confusions.head(20))
display(mistakes.head(20))

## 7. Матрица ошибок и следующие шаги

Матрица содержит все классы и поэтому может быть большой. В таблице выше показаны наиболее частые пары путаницы. Для production перед выкладкой дополнительно нужны отдельный финальный test, контроль качества входных данных, API и мониторинг; этот ноутбук отвечает за воспроизводимые эксперименты и сравнение моделей.

In [ ]:
matrix = confusion_matrix(y_val, best_pred, labels=labels)
fig, ax = plt.subplots(figsize=(12, 10))
image = ax.imshow(matrix, interpolation='nearest', cmap='Blues')
ticks = np.arange(0, len(labels), 5)
ax.set_xticks(ticks, [labels[i] for i in ticks], rotation=90, fontsize=7)
ax.set_yticks(ticks, [labels[i] for i in ticks], fontsize=7)
ax.set_xlabel('Предсказанный класс')
ax.set_ylabel('Истинный класс')
ax.set_title(f'Матрица ошибок на validation: {best_name}')
fig.colorbar(image, ax=ax, label='Число примеров')
fig.tight_layout()
matrix_path = OUT_DIR / 'notebook_best_model_confusion_matrix.png'
fig.savefig(matrix_path, dpi=160, bbox_inches='tight')
plt.show()
print(f'Матрица сохранена: {matrix_path}')
print(f'Модели сохранены в: {MODEL_DIR}')
print(f'MLflow database: {tracking_db}')